# Data Quality Assessment — Standalone Notebook

All pipeline logic is defined **inline** in this notebook.  
No ml-server source code required — runs anywhere with:
```
pip install pydantic numpy scipy pandas requests plotly
```

Sections:
1. **Inline pipeline code** (models + 4-step pipeline)
2. **Synthetic test** — controlled series with injected defects
3. **Parse-time variants** — ISO / Unix ms / empty-string defaults
4. **Batch assessment** — reads `inputs.csv` + `models.csv`, runs pipeline for every archive

In [21]:
import os, sys, json, pathlib, math, time, logging
from datetime import datetime, timedelta, timezone
from typing import Any, Dict, List, Optional, Tuple, Union

import numpy as np
import pandas as pd
import requests
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
from IPython.display import display

# SCADA / batch settings
SCADA_URL  = os.getenv('SCADA_URL', 'http://127.0.0.1:7080/api/v1/read/archives')
OBJECT_REF = os.getenv('OBJECT_REF', None)
STEP       = int(os.getenv('DQ_STEP', '3600'))

# Pipeline tuning
ALLOW_LOOK_AHEAD  = True
Z_SCORE_WINDOW    = 48
Z_SCORE_THRESHOLD = 3.0
STUCK_WINDOW      = 10

print('Standalone notebook loaded.')
print(f'SCADA URL : {SCADA_URL}')
print(f'Archive   : {OBJECT_REF}')
print(f'Step      : {STEP}s')

Standalone notebook loaded.
SCADA URL : http://127.0.0.1:7080/api/v1/read/archives
Archive   : None
Step      : 3600s


---
## 1 — Load Pipeline Module

Locates `lib/pipeline.py` in the ml-server directory tree and imports all classes.
No ml-server FastAPI stack needed — only `pydantic`, `numpy`, `scipy`.

In [22]:
import sys, pathlib

def _find_lib():
    current = pathlib.Path(os.path.abspath('.'))
    for _ in range(10):
        cand = current / 'lib' / 'pipeline.py'
        if cand.exists():
            lib_dir = str(current)
            if lib_dir not in sys.path:
                sys.path.insert(0, lib_dir)
            return lib_dir
        current = current.parent
    return None

_lib_root = _find_lib()
if _lib_root is None:
    raise ImportError(
        'Cannot locate lib/pipeline.py.  '
        'Run this notebook from inside the ml-server directory tree.')

from lib.pipeline import (  # noqa: E402
    _parse_time, _ts_to_iso, _dt_to_ms, _find_nan_runs,
    _step1_chronological, _step2_static_bounds, _step3_dynamic,
    _step4_impute, _compute_score,
    ScoringWeights, AssessRequest, AnomalyRecord, TagStats,
    MetricsScoring, Metadata, AssessResponse,
    DataQualityPipeline,
)

print(f'lib/pipeline.py loaded from: {_lib_root}')
print('  AssessRequest, DataQualityPipeline, AssessResponse -- all ready.')

lib/pipeline.py loaded from: /Users/rustamkrikbayev/Documents/projects/forecast/ml-server
  AssessRequest, DataQualityPipeline, AssessResponse -- all ready.


---
## 2 — Synthetic Pipeline Test

168-point hourly series with deliberate defects. Verifies every anomaly type is detected and labelled correctly.

In [23]:
N_POINTS  = 168
STEP_S    = 3600
NOW_UTC   = datetime.now(tz=timezone.utc).replace(minute=0, second=0, microsecond=0)
END_UTC   = NOW_UTC
START_UTC = END_UTC - timedelta(seconds=(N_POINTS - 1) * STEP_S)

rng    = np.random.default_rng(42)
t      = np.linspace(0, 4 * np.pi, N_POINTS)
signal = 300 + 60 * np.sin(t) + rng.normal(0, 5, N_POINTS)

grid_ms = np.array([
    int((START_UTC + timedelta(seconds=i * STEP_S)).timestamp() * 1000)
    for i in range(N_POINTS)
], dtype=np.int64)

# Inject defects
missing_idx = list(range(30, 36))   # 6-point gap
spike_val   = signal.copy()
spike_val[50]    = 50_000.0         # spike
spike_val[80:96] = 300.0            # stuck signal (16 pts)
spike_val[120]   = float('inf')     # hard limit
dup_ts = int(grid_ms[10])

raw_series = [[int(ts), float(v)] for ts, v in zip(grid_ms, spike_val)]
raw_series.insert(11, [dup_ts, float(spike_val[10]) + 20.0])  # duplicate
raw_series = [pt for pt in raw_series
              if pt[0] not in [int(grid_ms[j]) for j in missing_idx]]

print(f'Synthetic: {len(raw_series)} raw points  (expected on grid: {N_POINTS})')
print(f'Window : {START_UTC:%Y-%m-%d %H:%M UTC} -- {END_UTC:%Y-%m-%d %H:%M UTC}')

Synthetic: 163 raw points  (expected on grid: 168)
Window : 2026-05-22 10:00 UTC -- 2026-05-29 09:00 UTC


In [24]:
req_synth = AssessRequest(
    **{'from': START_UTC.strftime('%Y-%m-%dT%H:%M:%SZ')},
    object_ref='SYNTHETIC_TAG',
    to=END_UTC.strftime('%Y-%m-%dT%H:%M:%SZ'),
    step=STEP_S,
    allow_look_ahead=ALLOW_LOOK_AHEAD,
    z_score_window=Z_SCORE_WINDOW,
    z_score_threshold=Z_SCORE_THRESHOLD,
    stuck_window=STUCK_WINDOW,
)

pipeline = DataQualityPipeline(req_synth)
result   = pipeline.run({'SYNTHETIC_TAG': raw_series})

stats = result.metrics_scoring.tags['SYNTHETIC_TAG']
print(f'Quality score  : {result.metrics_scoring.overall_quality_score:.1f} / 100')
print(f'Expected points: {stats.total_expected_points}')
print(f'Missing        : {stats.missing_points_count}')
print(f'Duplicates     : {stats.duplicates_count}')
print(f'Spikes         : {stats.outliers_count}')
print(f'Stuck seqs     : {stats.stuck_sequences_count}')
print(f'Rate-of-change : {stats.rate_of_change_count}')
print(f'Long gaps      : {stats.long_gaps_count}')
print(f'Anomaly records: {len(result.anomalies_log)}')

print('\nSanity checks:')
for label, cond in [
    ('missing detected',       stats.missing_points_count >= len(missing_idx)),
    ('duplicate detected',     stats.duplicates_count >= 1),
    ('spike detected',         stats.outliers_count >= 1),
    ('stuck signal detected',  stats.stuck_sequences_count >= 1),
    ('score < 100',            result.metrics_scoring.overall_quality_score < 100),
    ('cleaned_data non-empty', len(result.cleaned_data) > 0),
]:
    print(f"  {'OK' if cond else 'FAIL'}  {label}")

Quality score  : 86.9 / 100
Expected points: 168
Missing        : 6
Duplicates     : 1
Spikes         : 1
Stuck seqs     : 15
Rate-of-change : 1
Long gaps      : 0
Anomaly records: 29

Sanity checks:
  OK  missing detected
  OK  duplicate detected
  OK  spike detected
  OK  stuck signal detected
  OK  score < 100
  OK  cleaned_data non-empty


In [25]:
ANOM_COLORS = {
    'missing': '#3498db', 'duplicate': '#9b59b6', 'hard_limit': '#e74c3c',
    'spike_outlier': '#e67e22', 'stuck_signal': '#f39c12', 'rate_of_change': '#1abc9c',
}

df_cleaned = pd.DataFrame(result.cleaned_data)
df_cleaned['ts']    = pd.to_datetime(df_cleaned['timestamp'], utc=True).dt.tz_convert(None)
df_cleaned['value'] = df_cleaned['SYNTHETIC_TAG'].astype(float)

df_raw = pd.DataFrame(raw_series, columns=['ts_ms', 'value_raw'])
df_raw['ts']        = pd.to_datetime(df_raw['ts_ms'], unit='ms', utc=True).dt.tz_convert(None)
df_raw['value_raw'] = df_raw['value_raw'].replace([float('inf'), float('-inf')], float('nan'))

df_anom = pd.DataFrame([a.model_dump() for a in result.anomalies_log])
df_anom['ts'] = pd.to_datetime(df_anom['timestamp'], utc=True).dt.tz_convert(None)

fig = make_subplots(rows=2, cols=1, shared_xaxes=True,
    subplot_titles=('Raw vs Cleaned', 'Anomaly types'),
    row_heights=[0.65, 0.35], vertical_spacing=0.08)

fig.add_trace(go.Scatter(x=df_raw['ts'], y=df_raw['value_raw'],
    mode='lines', name='Raw', opacity=0.45, line=dict(color='#95a5a6', width=1)), row=1, col=1)
fig.add_trace(go.Scatter(x=df_cleaned['ts'], y=df_cleaned['value'],
    mode='lines', name='Cleaned', line=dict(color='#2ecc71', width=2)), row=1, col=1)

for atype, color in ANOM_COLORS.items():
    sub = df_anom[df_anom['anomaly_type'] == atype]
    if sub.empty:
        continue
    sub_m = sub.merge(df_cleaned[['ts', 'value']], on='ts', how='left')
    fig.add_trace(go.Scatter(x=sub_m['ts'], y=sub_m['value'], mode='markers',
        name=atype, marker=dict(color=color, size=8, symbol='x')), row=1, col=1)

atype_vals = {'missing': 1, 'duplicate': 2, 'hard_limit': 3,
              'spike_outlier': 4, 'stuck_signal': 5, 'rate_of_change': 6}
for atype, yval in atype_vals.items():
    sub = df_anom[df_anom['anomaly_type'] == atype]
    if sub.empty:
        continue
    fig.add_trace(go.Scatter(x=sub['ts'], y=[yval]*len(sub), mode='markers',
        showlegend=False, marker=dict(color=ANOM_COLORS[atype], size=6)), row=2, col=1)
fig.update_yaxes(tickvals=list(atype_vals.values()), ticktext=list(atype_vals.keys()), row=2, col=1)
fig.update_layout(
    title=f'Synthetic Test  |  Score: {result.metrics_scoring.overall_quality_score:.1f}/100',
    hovermode='x unified', template='plotly_white', height=700,
    legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='right', x=1))
fig.show()

summary = df_anom.groupby(['anomaly_type', 'action_taken']).size().reset_index(name='count')
display(summary.sort_values('count', ascending=False).reset_index(drop=True))

,anomaly_type,action_taken,count
0,missing,cubic_interpolated,22
1,missing,linear_interpolated,2
2,duplicate,deduplicated,1
3,hard_limit,replaced_with_nan,1
4,rate_of_change,replaced_with_nan,1
5,spike_outlier,replaced_with_nan,1
6,stuck_signal,marked_as_nan_no_fill,1


---
## 3 — Parse-Time Variants

Verifies `from` / `to` accept ISO strings, Unix ms timestamps, and empty strings.

In [26]:
cases = [
    ('ISO string',       '2026-05-01T00:00:00Z'),
    ('ISO with offset',  '2026-05-01T05:00:00+05:00'),
    ('Unix ms (int)',    1746057600000),
    ('Unix ms (str)',    '1746057600000'),
    ('Empty string',    ''),
    ('None',            None),
]

print(f'{"Input":<30} {"Result":<35} Status')
print('-' * 75)
for label, value in cases:
    try:
        dt = _parse_time(value)
        result_str = dt.strftime('%Y-%m-%dT%H:%M:%SZ') if dt else 'None (default)'
        status = 'OK'
    except Exception as e:
        result_str = f'ERROR: {e}'
        status = 'FAIL'
    print(f'{label:<30} {result_str:<35} {status}')

print('\nEmpty from/to -> auto-default:')
req_empty = AssessRequest(**{'from': ''}, object_ref='T', to='', step=3600)
print(f'  start_time : {req_empty.start_time.strftime("%Y-%m-%dT%H:%M:%SZ")}')
print(f'  end_time   : {req_empty.end_time.strftime("%Y-%m-%dT%H:%M:%SZ")}')
print(f'  window     : {(req_empty.end_time - req_empty.start_time).total_seconds()/3600:.0f} h')

Input                          Result                              Status
---------------------------------------------------------------------------
ISO string                     2026-05-01T00:00:00Z                OK
ISO with offset                2026-05-01T05:00:00Z                OK
Unix ms (int)                  2025-05-01T00:00:00Z                OK
Unix ms (str)                  2025-05-01T00:00:00Z                OK
Empty string                   None (default)                      OK
None                           None (default)                      OK

Empty from/to -> auto-default:
  start_time : 2026-05-28T09:00:00Z
  end_time   : 2026-05-29T09:00:00Z
  window     : 24 h


---
## 4 — Batch Assessment from `inputs.csv`

Reads `local/models/training_workspace/models_enabled/inputs.csv` and `models.csv`,
joins on `object_ref`, fetches each archive from SCADA directly, and runs the inline pipeline.

| Column | Source | Meaning |
|--------|--------|---------|
| `input_ref` | inputs.csv | SCADA archive path to assess |
| `api_url` | inputs.csv | SCADA endpoint for that archive |
| `object_ref` | inputs.csv | Model reference (join key) |
| `step` | models.csv | Time step in seconds |
| `input_range` | models.csv | History window in steps |

**No server needed** -- fetches SCADA directly and runs the pipeline in-process.

In [27]:
def _find_enabled_dir():
    current = pathlib.Path(os.path.abspath('.'))
    for _ in range(10):
        cand = current / 'local' / 'models' / 'training_workspace' / 'models_enabled'
        if cand.is_dir():
            return cand
        cand2 = current / 'models' / 'training_workspace' / 'models_enabled'
        if cand2.is_dir():
            return cand2
        current = current.parent
    return None

ENABLED_DIR = _find_enabled_dir()
if ENABLED_DIR is None:
    raise FileNotFoundError(
        'Cannot locate models_enabled/ directory. '
        'Set ENABLED_DIR = pathlib.Path("/absolute/path") below.')

# Override here if needed:
# ENABLED_DIR = pathlib.Path('/absolute/path/to/models_enabled')

INPUTS_CSV = ENABLED_DIR / 'inputs.csv'
MODELS_CSV = ENABLED_DIR / 'models.csv'

BATCH_HOURS         = None   # None -> use input_range * step; or e.g. 168
BATCH_SCADA_TIMEOUT = 30     # seconds per archive

print(f'inputs.csv : {INPUTS_CSV}  (exists={INPUTS_CSV.exists()})')
print(f'models.csv : {MODELS_CSV}  (exists={MODELS_CSV.exists()})')

inputs.csv : /Users/rustamkrikbayev/Documents/projects/forecast/local/models/training_workspace/models_enabled/inputs.csv  (exists=True)
models.csv : /Users/rustamkrikbayev/Documents/projects/forecast/local/models/training_workspace/models_enabled/models.csv  (exists=True)


In [28]:
df_inputs = pd.read_csv(INPUTS_CSV, sep=';', comment='#',
                         names=['row_id','object_ref','pattern','input_ref','api_url'],
                         dtype=str).dropna(subset=['input_ref'])

df_models = pd.read_csv(MODELS_CSV, sep=';', comment='#',
                         names=['row_id','object_ref','input_range','output_range','step'],
                         dtype=str).dropna(subset=['object_ref'])
df_models['step']        = pd.to_numeric(df_models['step'],        errors='coerce').fillna(3600).astype(int)
df_models['input_range'] = pd.to_numeric(df_models['input_range'], errors='coerce').fillna(168).astype(int)

df_batch = (df_inputs
    .merge(df_models[['object_ref','step','input_range']], on='object_ref', how='left')
    .assign(step=lambda d: d['step'].fillna(3600).astype(int),
            input_range=lambda d: d['input_range'].fillna(168).astype(int))
    .reset_index(drop=True))

print(f'Loaded {len(df_batch)} input(s) to assess:')
display(df_batch[['row_id','object_ref','input_ref','step','input_range','api_url']])

Loaded 2 input(s) to assess:


,row_id,object_ref,input_ref,step,input_range,api_url
0,1214,/root/FP/PROJECT/AKMOLA/@regions/North Kazakhs...,/root/FP/PROJECT/AKMOLA/@regions/SevKaz/Load/P...,3600,360,http://127.0.0.1:7080/api/v1/read/archives
1,1170,/root/FP/PROJECT/AKMOLA/Nura_SES/@models/P_watt,/root/FP/PROJECT/AKMOLA/Nura_SES/Pgen_sum/arch...,3600,168,http://127.0.0.1:7080/api/v1/read/archives


In [29]:
from urllib.parse import urlparse, urlunparse

def _normalize_scada_url(url):
    parsed = urlparse(url)
    if parsed.hostname not in ('127.0.0.1', 'localhost'):
        return url
    if not pathlib.Path('/.dockerenv').exists():
        return url
    netloc = parsed.netloc.replace(parsed.hostname, 'host.docker.internal')
    return urlunparse(parsed._replace(netloc=netloc))


def _fetch_scada(api_url, archive, from_ms, to_ms, step_s, timeout=30):
    url  = _normalize_scada_url(api_url)
    body = {'from': from_ms, 'to': to_ms, 'archive': [archive], 'step': step_s}
    try:
        resp = requests.post(url, json=body, timeout=timeout)
        if resp.status_code == 200:
            data = resp.json()
            if isinstance(data, dict) and data:
                return data
    except Exception:
        pass
    return None


now_utc       = datetime.now(tz=timezone.utc).replace(minute=0, second=0, microsecond=0)
BATCH_RESULTS = []

for _, row in df_batch.iterrows():
    object_ref  = str(row['object_ref']).strip()
    input_ref   = str(row['input_ref']).strip()
    api_url     = str(row['api_url']).strip()
    step_s      = int(row['step'])
    input_range = int(row['input_range'])

    window_h = BATCH_HOURS if BATCH_HOURS else (input_range * step_s) // 3600
    to_dt    = now_utc
    from_dt  = to_dt - timedelta(hours=window_h)
    from_ms  = int(from_dt.timestamp() * 1000)
    to_ms    = int(to_dt.timestamp() * 1000)

    arch_ref_display = input_ref.replace('/root/FP/PROJECT/', '') if '/root/FP/PROJECT/' in input_ref else input_ref
    model_ref_display = object_ref.replace('/root/FP/PROJECT/', '') if '/root/FP/PROJECT/' in object_ref else object_ref
    short_model = object_ref.split('/')[-1] if '/' in object_ref else object_ref

    print(f'[{model_ref_display}]  arch={arch_ref_display}  window={window_h}h  step={step_s}s', end='  ')

    entry = dict(
        object_ref=object_ref, input_ref=input_ref,
        short_model=short_model, short_input=arch_ref_display,
        step_s=step_s, window_h=window_h,
        from_dt=from_dt.strftime('%Y-%m-%dT%H:%M:%SZ'),
        to_dt=to_dt.strftime('%Y-%m-%dT%H:%M:%SZ'),
        status='ok', score=None,
        n_expected=None, n_missing=None, n_dup=None,
        n_spikes=None, n_stuck=None, n_roc=None, n_long_gaps=None,
        n_raw_points=None, error=None, pipeline_result=None,
    )

    raw_payload = _fetch_scada(api_url, input_ref, from_ms, to_ms, step_s,
                               timeout=BATCH_SCADA_TIMEOUT)
    if raw_payload is None:
        entry.update(status='scada_unavailable', error=f'SCADA did not respond: {api_url}')
        print('SCADA unavailable')
        BATCH_RESULTS.append(entry)
        continue

    raw_series = raw_payload.get(input_ref, []) or next(iter(raw_payload.values()), [])
    entry['n_raw_points'] = len(raw_series)
    if not raw_series:
        entry.update(status='no_data', error='SCADA returned empty series')
        print('no data')
        BATCH_RESULTS.append(entry)
        continue

    try:
        req = AssessRequest(
            **{'from': entry['from_dt']},
            object_ref=input_ref, to=entry['to_dt'], step=step_s,
            allow_look_ahead=ALLOW_LOOK_AHEAD,
            z_score_window=Z_SCORE_WINDOW,
            z_score_threshold=Z_SCORE_THRESHOLD,
            stuck_window=STUCK_WINDOW,
        )
        res = DataQualityPipeline(req).run({input_ref: raw_series})
        st  = res.metrics_scoring.tags[input_ref]
        entry.update(
            score=res.metrics_scoring.overall_quality_score,
            n_expected=st.total_expected_points,
            n_missing=st.missing_points_count, n_dup=st.duplicates_count,
            n_spikes=st.outliers_count, n_stuck=st.stuck_sequences_count,
            n_roc=st.rate_of_change_count, n_long_gaps=st.long_gaps_count,
            pipeline_result=res,
        )
        badge = 'OK' if entry['score'] >= 80 else 'WARN' if entry['score'] >= 60 else 'LOW'
        print(f'{badge} score={entry["score"]:.1f}  miss={st.missing_points_count}')
    except Exception as exc:
        entry.update(status='pipeline_error', error=str(exc))
        print(f'ERROR: {exc}')

    BATCH_RESULTS.append(entry)

print(f'\nDone: {len(BATCH_RESULTS)} inputs assessed.')

[AKMOLA/@regions/North Kazakhstan/load/@models/P_watt]  arch=AKMOLA/@regions/SevKaz/Load/P_Load/archives/out_value  window=360h  step=3600s  OK score=99.0  miss=2
[AKMOLA/Nura_SES/@models/P_watt]  arch=AKMOLA/Nura_SES/Pgen_sum/archives/out_value  window=168h  step=3600s  OK score=98.2  miss=3

Done: 2 inputs assessed.


In [30]:
df_summary = pd.DataFrame([{
    'Model':      r['short_model'], 'Archive':    r['short_input'],
    'Step':       f"{r['step_s']}s", 'Window':  f"{r['window_h']}h",
    'Status':     r['status'],       'Score':     r['score'],
    'Expected':   r['n_expected'],   'Raw pts':   r['n_raw_points'],
    'Missing':    r['n_missing'],    'Duplicates': r['n_dup'],
    'Spikes':     r['n_spikes'],     'Stuck':     r['n_stuck'],
    'RoC':        r['n_roc'],        'Long gaps': r['n_long_gaps'],
    'Error':      r['error'],
} for r in BATCH_RESULTS])

def _color_score(val):
    if pd.isna(val):    return 'background-color: #f5b7b1'
    if val >= 80:       return 'background-color: #d5f5e3'
    if val >= 60:       return 'background-color: #fdebd0'
    return                     'background-color: #f5b7b1'

print('Batch quality assessment summary:')
display(df_summary.style
    .applymap(_color_score, subset=['Score'])
    .format({'Score': lambda v: f'{v:.1f}' if pd.notna(v) else '--'})
    .set_properties(**{'text-align': 'right'}))

Batch quality assessment summary:


/var/folders/z9/_xmzk2xd65b6z8w_nqdgdbkm0000gn/T/ipykernel_2661/1968626056.py:20: FutureWarning: Styler.applymap has been deprecated. Use Styler.map instead.
  .applymap(_color_score, subset=['Score'])


,Model,Archive,Step,Window,Status,Score,Expected,Raw pts,Missing,Duplicates,Spikes,Stuck,RoC,Long gaps,Error
0,P_watt,AKMOLA/@regions/SevKaz/Load/P_Load/archives/out_value,3600s,360h,ok,99.0,361,360,2,0,2,0,1,0,None
1,P_watt,AKMOLA/Nura_SES/Pgen_sum/archives/out_value,3600s,168h,ok,98.2,169,168,3,0,0,0,1,0,None


In [31]:
ok_rows = [r for r in BATCH_RESULTS if r['score'] is not None]

if not ok_rows:
    print('No successful assessments to visualise.')
else:
    labels  = [f"{r['short_model']}\n{r['short_input']}" for r in ok_rows]
    scores  = [r['score'] for r in ok_rows]
    colors  = ['#2ecc71' if s >= 80 else '#e67e22' if s >= 60 else '#e74c3c' for s in scores]

    fig_bar = go.Figure(go.Bar(x=labels, y=scores, marker_color=colors,
        text=[f'{s:.1f}' for s in scores], textposition='outside'))
    fig_bar.add_hline(y=80, line_dash='dash', line_color='#e74c3c',
                      annotation_text='threshold 80', annotation_position='top right')
    fig_bar.update_layout(title='Input Data Quality -- All Models',
        yaxis=dict(title='Score / 100', range=[0, 115]),
        xaxis_title='Model / Archive', template='plotly_white', height=420)
    fig_bar.show()

    defect_cols   = ['Missing', 'Duplicates', 'Spikes', 'Stuck', 'RoC']
    defect_keys   = ['n_missing', 'n_dup', 'n_spikes', 'n_stuck', 'n_roc']
    defect_colors = ['#3498db', '#9b59b6', '#e67e22', '#f39c12', '#1abc9c']
    fig_def = go.Figure()
    for col, key, color in zip(defect_cols, defect_keys, defect_colors):
        fig_def.add_trace(go.Bar(name=col, x=labels,
            y=[r.get(key) or 0 for r in ok_rows], marker_color=color))
    fig_def.update_layout(barmode='stack', title='Defect Breakdown -- All Models',
        xaxis_title='Model / Archive', yaxis_title='Point count',
        template='plotly_white', height=400)
    fig_def.show()

    completeness = [
        round(100 * (r['n_raw_points'] or 0) / r['n_expected'], 1)
        if r['n_expected'] else None
        for r in ok_rows
    ]
    fig_comp = go.Figure(go.Bar(x=labels, y=completeness,
        marker_color=['#2ecc71' if (v or 0) >= 95 else '#e67e22' if (v or 0) >= 80 else '#e74c3c'
                      for v in completeness],
        text=[f'{v}%' if v is not None else '--' for v in completeness], textposition='outside'))
    fig_comp.add_hline(y=95, line_dash='dash', line_color='#e74c3c',
                       annotation_text='95%', annotation_position='top right')
    fig_comp.update_layout(title='Raw Data Completeness (received vs expected)',
        yaxis=dict(title='%', range=[0, 115]),
        xaxis_title='Model / Archive', template='plotly_white', height=380)
    fig_comp.show()

In [32]:
ok_rows = [r for r in BATCH_RESULTS if r['pipeline_result'] is not None]

if not ok_rows:
    print('No pipeline results to plot.')
else:
    pal = ['#2ecc71','#e74c3c','#9b59b6','#f39c12','#1abc9c',
           '#3498db','#e67e22','#1a5276','#7d3c98','#117a65']
    fig_all = go.Figure()
    for i, r in enumerate(ok_rows):
        res   = r['pipeline_result']
        tag   = r['input_ref']
        df_cl = pd.DataFrame(res.cleaned_data)
        df_cl['ts'] = pd.to_datetime(df_cl['timestamp'], utc=True).dt.tz_convert(None)
        if tag not in df_cl.columns:
            continue
        fig_all.add_trace(go.Scatter(
            x=df_cl['ts'], y=df_cl[tag].astype(float), mode='lines',
            name=f"{r['short_model']} ({r['score']:.0f}/100)",
            line=dict(color=pal[i % len(pal)], width=1.6)))
    fig_all.update_layout(title='Cleaned Input Series -- All Archives',
        xaxis_title='Time (UTC)', yaxis_title='Value',
        hovermode='x unified', template='plotly_white', height=500,
        legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='right', x=1))
    fig_all.show()

In [ ]:
export_dir = pathlib.Path('.') / 'exports'
export_dir.mkdir(exist_ok=True)
ts_now = datetime.now().strftime('%Y%m%d_%H%M%S')

# fpath_summary = export_dir / f'batch_quality_summary__{ts_now}.csv'
# df_summary.to_csv(fpath_summary, index=False)
# print(f'Summary CSV   -> {fpath_summary}')

for r in BATCH_RESULTS:
    if r['pipeline_result'] is None:
        continue
    alog = r['pipeline_result'].anomalies_log
    if not alog:
        continue
    slug    = r['short_input'][:40].replace(' ', '_')
    fpath_a = export_dir / f'batch_anomalies__{slug}__{ts_now}.csv'
    os.makedirs(fpath_a.parent, exist_ok=True)
    pd.DataFrame([a.model_dump() for a in alog]).to_csv(fpath_a, index=False)
    print(f'Anomaly log   -> {fpath_a}  ({len(alog)} records)')

batch_report = [{
    'object_ref': r['object_ref'], 'input_ref': r['input_ref'],
    'from': r['from_dt'], 'to': r['to_dt'], 'step_s': r['step_s'],
    'status': r['status'], 'score': r['score'],
    'stats': {k: r[k] for k in ('n_expected','n_raw_points','n_missing',
                                  'n_dup','n_spikes','n_stuck','n_roc','n_long_gaps')},
    'error': r['error'],
} for r in BATCH_RESULTS]

fpath_json = export_dir / f'batch_quality_report__{ts_now}.json'
with open(fpath_json, 'w') as f:
    json.dump(batch_report, f, indent=2, ensure_ascii=False, default=str)
print(f'Batch report  -> {fpath_json}')

Summary CSV   -> exports/batch_quality_summary__20260529_143545.csv
Anomaly log   -> exports/batch_anomalies__AKMOLA/@regions/SevKaz/Load/P_Load/archi__20260529_143545.csv  (8 records)
Anomaly log   -> exports/batch_anomalies__AKMOLA/Nura_SES/Pgen_sum/archives/out_va__20260529_143545.csv  (5 records)
Batch report  -> exports/batch_quality_report__20260529_143545.json
